# MMUbiPred–Context Residual Hybrid: fixed-30-epoch exploratory rebuild

This notebook performs a separate post-test exploratory fixed-duration rebuild. In every one of five protein-grouped outer folds, both the MMUbiPred-compatible 49-residue local expert and the 257-residue LoRA-ESM2 context expert train for **exactly 30 complete epochs** on the whole outer-training partition. There is no inner validation, early stopping, or best-epoch selection. Epoch-30 out-of-fold predictions fit a new nonnegative residual stacker. A final section refits both components for exactly 30 epochs on all 89,551 context-valid released training records.

The current frozen hybrid remains unchanged. This new run is permanently exploratory because the historical and external tests were already inspected. No independent or external test set is loaded by this notebook. Expected compute is approximately 35–45 GPU hours, depending on the assigned GPU and Drive performance.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Private-repository access uses a fine-grained read-only GitHub token stored in Colab Secrets as `GITHUB_TOKEN`. An optional `HF_TOKEN` supports authenticated model retrieval. The Drive-backed Hugging Face cache avoids repeated ESM-2 downloads when it is copied or shared across accounts.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN, HF_TOKEN = secret('GITHUB_TOKEN'), secret('HF_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = '/content/drive/MyDrive/MMUbiPred-research/huggingface-cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip uninstall -y torchao

In [ ]:
import json, torch

if not torch.cuda.is_available():
    raise RuntimeError('A GPU runtime is required for training.')
print('GPU:', torch.cuda.get_device_name(0))

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

## Protocol and Drive preflight

Only the UniProt training-sequence cache is required from earlier experiments. The new run has its own directory and cannot overwrite the frozen 5/7-epoch hybrid. Exact epoch-boundary continuation requires `num_workers = 0`, which is locked in the configuration.

In [ ]:
CONFIG = Path('experiment/configs/mmubipred_context_residual_fixed30_exploratory_v1.json')
RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-residual-fixed30-exploratory-v1')
FULL_REFIT_DIR = RUN_DIR / 'full_refit'
SEQUENCE_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/context/uniprot_training_sequences.json')

required_files = {
    'fixed30 configuration': CONFIG,
    'outer-fold training script': Path('experiment/scripts/run_context_residual_fixed30_cv.py'),
    'full-refit script': Path('experiment/scripts/refit_context_residual_fixed30.py'),
    'fixed30 protocol module': Path('experiment/src/ubipred/fixed30.py'),
    'positive released training FASTA': Path('replication/MMUbiPred/Positive_90_percent_training_set_DeepUBI.fasta'),
    'negative released training FASTA': Path('replication/MMUbiPred/Negative_90_percent_training_set_DeepUBI.fasta'),
    'AAindex table': Path('replication/MMUbiPred/aaindex31.txt'),
    'UniProt training-sequence cache': SEQUENCE_CACHE,
}
missing = []
for label, path in required_files.items():
    valid = path.is_file() and path.stat().st_size > 0
    print('FOUND  ' if valid else 'MISSING', f'{label}:', path)
    if not valid:
        missing.append(f'{label}: {path}')
if missing:
    raise FileNotFoundError('Required training artifacts are missing:\n' + '\n'.join(missing))

protocol = json.loads(CONFIG.read_text())
assert protocol['fixed_epochs'] == 30
assert protocol['num_workers'] == 0
assert protocol['confirmatory_claim_allowed'] is False
assert Path(protocol['output_dir']) == RUN_DIR, 'Configuration and notebook output paths differ'
assert Path(protocol['sequence_cache']) == SEQUENCE_CACHE, 'Configuration and notebook cache paths differ'
RUN_DIR.mkdir(parents=True, exist_ok=True)
write_probe = RUN_DIR / '.colab_write_probe'
write_probe.write_text('ok')
write_probe.unlink()
print('Run directory:', RUN_DIR)
print('Drive write check: PASSED')
print('Fixed epochs per expert per fold:', protocol['fixed_epochs'])
print('Early stopping: disabled by the fixed-refit training engine')

In [ ]:
run_live([
    sys.executable, '-u', '-m', 'unittest', 'discover',
    '-s', 'experiment/tests',
    '-p', 'test_context_residual_fixed30_protocol.py',
])

## Resume-status audit

This audit is re-executable after a session reconnects. `outer_predictions.npz` denotes a completed expert-fold; otherwise, `refit_resume.pt` records the last completed epoch.

In [ ]:
def resume_status(run_dir):
    rows = []
    for fold in range(5):
        for expert in ('local', 'context'):
            expert_dir = run_dir / 'folds' / f'fold_{fold}' / expert
            prediction = expert_dir / 'outer_predictions.npz'
            resume_path = expert_dir / 'refit_resume.pt'
            if prediction.exists():
                status, epoch = 'complete', 30
            elif resume_path.exists():
                checkpoint = torch.load(resume_path, map_location='cpu', weights_only=False)
                status, epoch = 'resumable', int(checkpoint['completed_epoch'])
                del checkpoint
            else:
                status, epoch = 'not started', 0
            rows.append({'fold': fold, 'expert': expert, 'status': status, 'completed_epoch': epoch})
    return rows

display(resume_status(RUN_DIR))

## Run or resume all outer folds automatically

This cell is invariant across sessions and Colab accounts. The script scans folds 0–4 in order, skips completed experts, resumes an interrupted expert after its latest completed epoch, and continues automatically through the remaining experts and folds. No fold variable requires manual selection. Cross-account continuation requires the complete run directory at the same Drive path, followed by the setup cells and this automatic fold runner.

In [ ]:
run_live([
    sys.executable, '-u',
    'experiment/scripts/run_context_residual_fixed30_cv.py',
    '--config', CONFIG,
    '--output-dir', RUN_DIR,
    '--resume',
    '--allow-post-test-exploration',
])

## Inspect the fixed-30-epoch OOF comparison

The summary appears only after all ten final-epoch expert-fold predictions exist. This out-of-fold evidence is the sole basis for assessing the 30-epoch hybrid's development performance. No test set is accessed.

In [ ]:
SUMMARY_PATH = RUN_DIR / 'summary.json'
if not SUMMARY_PATH.exists():
    display(resume_status(RUN_DIR))
    completed = list(RUN_DIR.glob('folds/fold_*/*/outer_predictions.npz'))
    print(f'Completed expert-fold predictions: {len(completed)}/10')
else:
    fixed30_summary = json.loads(SUMMARY_PATH.read_text())
    print('Comparison valid:', fixed30_summary['comparison_valid'])
    print('Fixed epochs:', fixed30_summary['fixed_epochs_per_expert_per_fold'])
    print('Early stopping:', fixed30_summary['early_stopping'])
    display(fixed30_summary['metrics'])
    display(fixed30_summary['stack_minus_context_fixed_threshold'])
    display(fixed30_summary['final_stacker_for_future_refit'])
    print('Independent/external test accessed:', fixed30_summary['independent_or_external_test_accessed'])

## Full-data refit of both components

The full-data refit is enabled only when `summary.json` exists and `comparison_valid` is true. The local expert and then the context expert train on all context-valid released training records for exactly 30 epochs. Residual weights derive exclusively from the fixed-30-epoch OOF predictions. There is no validation, early stopping, threshold search, or test access. This section is resumable after every epoch.

In [ ]:
if not SUMMARY_PATH.exists():
    raise FileNotFoundError('Complete all five folds before the full refit.')
fixed30_summary = json.loads(SUMMARY_PATH.read_text())
if not fixed30_summary['comparison_valid']:
    raise RuntimeError('A collapsed expert invalidated the OOF comparison; full-data refit is not authorized.')
run_live([
    sys.executable, '-u',
    'experiment/scripts/refit_context_residual_fixed30.py',
    '--config', CONFIG,
    '--development-run-dir', RUN_DIR,
    '--output-dir', FULL_REFIT_DIR,
    '--resume',
    '--allow-post-test-exploration',
])

In [ ]:
FINAL_SUMMARY_PATH = FULL_REFIT_DIR / 'hybrid_refit_summary.json'
if FINAL_SUMMARY_PATH.exists():
    final_refit = json.loads(FINAL_SUMMARY_PATH.read_text())
    display(final_refit)
else:
    for expert in ('local', 'context'):
        resume_path = FULL_REFIT_DIR / expert / 'refit_resume.pt'
        if resume_path.exists():
            checkpoint = torch.load(resume_path, map_location='cpu', weights_only=False)
            print(expert, 'completed epoch:', checkpoint['completed_epoch'], '/30')
            del checkpoint
        else:
            print(expert, 'not started')

## Moving to another Colab account

Cross-account continuation requires the **entire** `mmubipred-context-residual-fixed30-exploratory-v1` directory and `context/uniprot_training_sequences.json` at the corresponding paths under `MyDrive/MMUbiPred-research`; `best.pt` alone is insufficient. The run directory contains the optimizer, AMP scaler, RNG, DataLoader generator, history, fold assignments, and manifest required by `--resume`. The optional `MMUbiPred-research/huggingface-cache` avoids another ESM-2 download. After the setup and preflight cells, the resume audit and automatic all-fold runner locate the unfinished work without manual fold selection.

In [ ]:
transfer_items = [
    RUN_DIR,
    SEQUENCE_CACHE,
    Path('/content/drive/MyDrive/MMUbiPred-research/huggingface-cache'),
]
for path in transfer_items:
    print('COPY' if path.exists() else 'OPTIONAL/MISSING', path)